[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/fastapi-certified/notebooks/day-13-deployment-docker.ipynb#scrollTo=a1b2c3d4)

---
# Day 13 · Deployment — Uvicorn, Gunicorn, and Docker
**certified-journeys / fastapi-certified** · Review

> **Goal for today:** Understand how to deploy a FastAPI app with Uvicorn and Gunicorn, configure it with environment variables via pydantic-settings, add a health endpoint, and know what a production Dockerfile looks like — all without actually running a server process.


In [ ]:
%pip install -q fastapi httpx pydantic-settings python-dotenv


---
## Step 1 · Uvicorn and ASGI — the runtime model

FastAPI is an **ASGI** framework. It needs an ASGI server to handle HTTP connections. The two standard choices are:

| Server | Role | Use in production? |
|---|---|---|
| **Uvicorn** | Pure ASGI server — fast, async | Yes — single process or behind Gunicorn |
| **Gunicorn** | Process manager — forks workers | Yes — manages multiple Uvicorn workers |
| **`UvicornWorker`** | Gunicorn worker class that uses Uvicorn | Yes — combine both strengths |

**Single-container deployment (e.g., Kubernetes):** run one Uvicorn process, let the orchestrator scale horizontally.

**VM / bare-metal deployment:** run Gunicorn with `UvicornWorker` — `workers = 2 × CPU_cores + 1`.

### Key Uvicorn CLI flags

```bash
uvicorn app.main:app \
  --host 0.0.0.0 \
  --port 8000 \
  --workers 1 \
  --log-level info
```

### Gunicorn + UvicornWorker

```bash
gunicorn app.main:app \
  -k uvicorn.workers.UvicornWorker \
  --workers 5 \
  --bind 0.0.0.0:8000
```


In [ ]:
# We demonstrate FastAPI apps with TestClient — no live server process needed in Colab.
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI(title="Deployment Demo", version="1.0.0")


@app.get("/")
def root():
    return {"message": "Hello from FastAPI"}


# TestClient runs the ASGI app in-process — no uvicorn.run() needed
client = TestClient(app)

response = client.get("/")
print(response.status_code, response.json())

# Check the auto-generated OpenAPI schema is accessible
schema_resp = client.get("/openapi.json")
print("OpenAPI title:", schema_resp.json()["info"]["title"])


**What just happened?**
- `TestClient` wraps the ASGI app and dispatches requests in-process — perfect for notebooks and CI
- FastAPI automatically generates `/openapi.json` and `/docs` with no extra code
- **In production** you'd replace `TestClient` calls with `uvicorn app.main:app --host 0.0.0.0 --port 8000`


---
## Step 2 · Environment-based configuration with pydantic-settings

Hard-coding config values is an anti-pattern. `pydantic-settings` provides `BaseSettings` — a Pydantic model that reads values from environment variables (or a `.env` file) automatically.

| Source | Priority |
|---|---|
| Environment variable (`export HOST=0.0.0.0`) | Highest |
| `.env` file in working directory | Second |
| Default value in the model | Lowest |

The name matching is **case-insensitive**: `APP_ENV` in the environment → `app_env` field in the model.

### Why this matters for deployment

- Twelve-Factor App principle: config in the environment, not in code
- Docker: pass `-e APP_ENV=production` at run time
- Kubernetes: inject via `ConfigMap` or `Secret`


In [ ]:
import os
from pydantic_settings import BaseSettings
from pydantic import Field


class Settings(BaseSettings):
    """App configuration — values read from environment or .env file."""

    # Each field maps to an env var of the same name (case-insensitive)
    app_name: str = Field(default="FastAPI App", description="Display name")
    app_env: str = Field(default="development", description="development | staging | production")
    host: str = Field(default="127.0.0.1", description="Bind host")
    port: int = Field(default=8000, description="Bind port")
    log_level: str = Field(default="info", description="Uvicorn log level")
    workers: int = Field(default=1, description="Number of Uvicorn worker processes")
    debug: bool = Field(default=False, description="Enable debug mode")

    class Config:
        # Reads from .env file if present; environment vars override
        env_file = ".env"
        env_file_encoding = "utf-8"


# Simulate environment variable injection (what Docker / Kubernetes would do)
os.environ["APP_ENV"] = "production"
os.environ["WORKERS"] = "3"
os.environ["LOG_LEVEL"] = "warning"

settings = Settings()

print(f"App name : {settings.app_name}")
print(f"App env  : {settings.app_env}")   # 'production' from env
print(f"Host     : {settings.host}")       # default
print(f"Port     : {settings.port}")       # default
print(f"Workers  : {settings.workers}")    # 3 from env
print(f"Log level: {settings.log_level}")  # 'warning' from env
print(f"Debug    : {settings.debug}")      # default False


**What just happened?**
- `BaseSettings` scanned environment variables and overrode the defaults where matches were found
- **`APP_ENV=production`** and **`WORKERS=3`** came from `os.environ`, simulating Docker `-e` flags
- **`host`** and **`port`** kept their defaults because no env var was set
- Pydantic validates types automatically — `WORKERS=abc` would raise a `ValidationError`


---
## Step 3 · Integrating settings into the FastAPI app

The `lru_cache` pattern avoids re-reading environment variables on every request. `get_settings()` is called once, cached, and injected via FastAPI's dependency system.

```python
# app/config.py
from functools import lru_cache
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    app_env: str = "development"
    ...

@lru_cache
def get_settings() -> Settings:
    return Settings()
```

```python
# app/main.py
from fastapi import Depends
from app.config import Settings, get_settings

@app.get("/info")
def info(settings: Settings = Depends(get_settings)):
    return {"env": settings.app_env, "workers": settings.workers}
```


In [ ]:
from functools import lru_cache
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient
from pydantic_settings import BaseSettings
from pydantic import Field


class AppSettings(BaseSettings):
    app_env: str = Field(default="development")
    app_name: str = Field(default="FastAPI App")
    workers: int = Field(default=1)
    log_level: str = Field(default="info")

    class Config:
        env_file = ".env"


@lru_cache
def get_settings() -> AppSettings:
    # Called once; result cached for the lifetime of the process
    return AppSettings()


app2 = FastAPI(title="Settings Demo")


@app2.get("/info")
def info(settings: AppSettings = Depends(get_settings)):
    """Return runtime configuration — useful for debugging deployments."""
    return {
        "app_name": settings.app_name,
        "app_env": settings.app_env,
        "workers": settings.workers,
        "log_level": settings.log_level,
    }


client2 = TestClient(app2)
resp = client2.get("/info")
print(resp.status_code)
import json
print(json.dumps(resp.json(), indent=2))


**What just happened?**
- `@lru_cache` ensures `Settings()` is instantiated exactly once — no re-reading on every request
- `Depends(get_settings)` injects the cached settings object into the route function
- **In tests** you can override `get_settings` with `app.dependency_overrides[get_settings] = lambda: TestSettings(...)` to inject test values


---
## Step 4 · Health endpoint for load balancer probes

Load balancers and container orchestrators (Kubernetes, ECS) send **readiness probes** to determine whether a pod is healthy. The convention is a `GET /health` endpoint that returns:
- **`200 OK`** when the app is ready to serve traffic
- **`503 Service Unavailable`** when it's not (e.g., database is down)

### Two types of probes (Kubernetes)

| Probe | When used | Action on failure |
|---|---|---|
| **Liveness** | Is the app still alive? | Restart the pod |
| **Readiness** | Is the app ready for traffic? | Remove from load balancer |

A good health endpoint checks **downstream dependencies** (DB, cache) and returns their status.


In [ ]:
import time
from fastapi import FastAPI, Response, status
from fastapi.testclient import TestClient
from pydantic import BaseModel

# Track startup time to report uptime
START_TIME = time.time()

app3 = FastAPI(title="Health Check Demo")


class HealthResponse(BaseModel):
    status: str          # "ok" | "degraded" | "down"
    uptime_seconds: float
    checks: dict


def check_database() -> dict:
    """Simulate a DB connectivity check. Replace with real DB ping in production."""
    # In production: try a cheap query like `SELECT 1`
    return {"status": "ok", "latency_ms": 2.1}


def check_cache() -> dict:
    """Simulate a cache (Redis) connectivity check."""
    return {"status": "ok", "latency_ms": 0.4}


@app3.get("/health", response_model=HealthResponse)
def health(response: Response):
    """Readiness probe — returns 200 when all checks pass, 503 otherwise."""
    db = check_database()
    cache = check_cache()

    all_ok = db["status"] == "ok" and cache["status"] == "ok"

    # Set HTTP status code based on health
    if not all_ok:
        response.status_code = status.HTTP_503_SERVICE_UNAVAILABLE

    return HealthResponse(
        status="ok" if all_ok else "degraded",
        uptime_seconds=round(time.time() - START_TIME, 2),
        checks={"database": db, "cache": cache},
    )


@app3.get("/health/live")
def liveness():
    """Liveness probe — just checks the process is running (no dependency checks)."""
    return {"status": "ok"}


client3 = TestClient(app3)

# Test the health endpoint
resp = client3.get("/health")
print(f"Status code: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

# Test liveness (simpler — no dep checks)
live_resp = client3.get("/health/live")
print(f"\nLiveness: {live_resp.status_code} — {live_resp.json()}")


**What just happened?**
- `/health` returns `200` with a structured payload including sub-checks — load balancers read the status code
- `/health/live` is intentionally lightweight — liveness probes run frequently and shouldn't query the DB
- **`response.status_code = 503`** lets FastAPI set the HTTP status while still returning the JSON body


---
## Step 5 · The production Dockerfile

Docker packages the app and all its dependencies into an image that runs identically in dev, staging, and production.

**Dockerfile best practices:**
- Use `python:3.12-slim` — smaller attack surface than full Debian
- `COPY requirements.txt` before `COPY app/` — Docker layer caching means pip install only re-runs when `requirements.txt` changes
- `--no-cache-dir` — saves image space
- Run as non-root user — security hardening
- `CMD` with exec form (JSON array) — signals go to the process directly, not a shell

The Dockerfile is shown as a string — Docker commands must be run in a Docker environment, not Colab.


In [ ]:
# Production Dockerfile — printed here for reference, not executed in Colab
dockerfile_content = '''\
# syntax=docker/dockerfile:1
FROM python:3.12-slim

# Set working directory
WORKDIR /app

# Install system deps (if any) — keep this layer separate for caching
# RUN apt-get update && apt-get install -y --no-install-recommends gcc && rm -rf /var/lib/apt/lists/*

# Copy requirements first — Docker will cache this layer unless requirements change
COPY requirements.txt .

# Install Python dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Copy application source code
COPY app/ ./app/

# Create a non-root user for security
RUN adduser --disabled-password --gecos "" appuser
USER appuser

# Expose the port uvicorn will listen on
EXPOSE 8000

# Health check for Docker itself (optional — Kubernetes has its own probes)
HEALTHCHECK --interval=30s --timeout=5s --start-period=10s --retries=3 \\
  CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"

# Use exec form (JSON array) so signals (SIGTERM) reach uvicorn, not a shell
# Single worker — let Kubernetes handle horizontal scaling
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"]
'''

print(dockerfile_content)

# Corresponding requirements.txt
requirements = '''\
fastapi>=0.111.0
uvicorn[standard]>=0.29.0
pydantic-settings>=2.2.0
python-dotenv>=1.0.0
'''

print("=== requirements.txt ===")
print(requirements)


**What just happened?**
- The Dockerfile follows the **layer caching** pattern: stable layers (OS, dependencies) come first
- **Non-root user** is a security requirement in many production environments
- **Exec form CMD** (`["uvicorn", ...]`) vs shell form (`CMD uvicorn ...`) — exec form means Docker `stop` sends SIGTERM directly to uvicorn, enabling graceful shutdown


---
## Step 6 · Docker build and run commands

These are the commands you'd run in a terminal with Docker installed. They're shown here as strings — run them in your local terminal or CI pipeline.

### Build

```bash
# Build the image and tag it 'myapi'
docker build -t myapi .

# Build with a specific tag (useful for versioning)
docker build -t myapi:1.0.0 .
```

### Run

```bash
# Run in foreground — map host port 8000 to container port 8000
docker run -p 8000:8000 myapi

# Run in background (-d), inject env vars (-e), name the container
docker run -d \
  --name myapi-prod \
  -p 8000:8000 \
  -e APP_ENV=production \
  -e LOG_LEVEL=warning \
  myapi

# Check logs
docker logs myapi-prod -f

# Stop gracefully
docker stop myapi-prod
```

### Multi-stage build (advanced — smaller images)

```dockerfile
# Stage 1: install deps in a build image
FROM python:3.12-slim AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir --target=/app/deps -r requirements.txt

# Stage 2: copy only what's needed into the final image
FROM python:3.12-slim
WORKDIR /app
COPY --from=builder /app/deps /app/deps
COPY app/ ./app/
ENV PYTHONPATH=/app/deps
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
```


In [ ]:
# Putting it all together: a minimal but production-ready FastAPI app
# This is what would live in app/main.py in your Docker image

import time
import json
from functools import lru_cache
from fastapi import FastAPI, Depends, Response, status
from fastapi.testclient import TestClient
from pydantic import BaseModel
from pydantic_settings import BaseSettings


# ── Config ────────────────────────────────────────────────────────────
class ProductionSettings(BaseSettings):
    app_name: str = "FastAPI Production App"
    app_env: str = "development"
    log_level: str = "info"
    workers: int = 1

    class Config:
        env_file = ".env"


@lru_cache
def get_settings() -> ProductionSettings:
    return ProductionSettings()


# ── App ───────────────────────────────────────────────────────────────
BOOT_TIME = time.time()
production_app = FastAPI(
    title="My Production API",
    version="1.0.0",
    docs_url="/docs",      # Swagger UI
    redoc_url="/redoc",    # ReDoc
    openapi_url="/openapi.json",
)


class HealthCheck(BaseModel):
    status: str
    environment: str
    uptime_seconds: float


@production_app.get("/health", response_model=HealthCheck, tags=["ops"])
def health_check(
    response: Response,
    settings: ProductionSettings = Depends(get_settings),
):
    """Readiness probe for load balancers and Kubernetes."""
    return HealthCheck(
        status="ok",
        environment=settings.app_env,
        uptime_seconds=round(time.time() - BOOT_TIME, 3),
    )


@production_app.get("/", tags=["root"])
def root(settings: ProductionSettings = Depends(get_settings)):
    return {
        "service": settings.app_name,
        "env": settings.app_env,
        "docs": "/docs",
    }


# ── Tests ─────────────────────────────────────────────────────────────
prod_client = TestClient(production_app)

root_resp = prod_client.get("/")
health_resp = prod_client.get("/health")

print("Root:", root_resp.status_code, root_resp.json())
print("Health:", health_resp.status_code)
print(json.dumps(health_resp.json(), indent=2))

# Assert the health endpoint returns 200
assert health_resp.status_code == 200, "Health check must return 200"
assert health_resp.json()["status"] == "ok"
print("\nAll assertions passed!")


**What just happened?**
- The complete production-ready app: settings injection, health endpoint, proper tags for OpenAPI grouping
- `tags=["ops"]` groups the health endpoint in its own section in the Swagger UI
- `docs_url`/`redoc_url` are explicitly set — in some deployments you'd disable them in production (`docs_url=None`)


---
## Step 7 · HTTPS and reverse proxy

FastAPI (and Uvicorn) do **not** handle TLS termination in production. Instead, a reverse proxy sits in front:

```
Internet → Nginx (TLS termination) → Uvicorn (HTTP) → FastAPI app
```

| Reverse proxy | Use case |
|---|---|
| **Nginx** | VM / bare-metal deployments |
| **Traefik** | Docker / Kubernetes — auto-discovers services, handles Let's Encrypt |
| **AWS ALB** | AWS ECS / EKS — offloads TLS, does path-based routing |
| **GCP Load Balancer** | GCP GKE |

### Nginx config snippet

```nginx
server {
    listen 443 ssl;
    server_name api.example.com;

    ssl_certificate     /etc/letsencrypt/live/api.example.com/fullchain.pem;
    ssl_certificate_key /etc/letsencrypt/live/api.example.com/privkey.pem;

    location / {
        proxy_pass http://127.0.0.1:8000;  # Forward to Uvicorn
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
    }
}
```

### `root_path` for sub-path deployments

If your app is served at `/api/v1/` (not `/`), tell FastAPI:

```python
app = FastAPI(root_path="/api/v1")
# or set via uvicorn: uvicorn app.main:app --root-path /api/v1
```


In [ ]:
# Demonstrate root_path — changes how OpenAPI docs resolve URLs
from fastapi import FastAPI
from fastapi.testclient import TestClient


# Simulating deployment at /api/v1 behind a reverse proxy
sub_path_app = FastAPI(
    title="Sub-path Deployment Demo",
    root_path="/api/v1",  # The prefix the reverse proxy strips
)


@sub_path_app.get("/users")
def list_users():
    return [{"id": 1, "name": "Alice"}]


@sub_path_app.get("/health")
def health():
    return {"status": "ok"}


sub_client = TestClient(sub_path_app)

# Routes are still accessed at their path — root_path affects OpenAPI server URL
resp = sub_client.get("/users")
print("Users:", resp.status_code, resp.json())

# Check the OpenAPI schema — servers[0].url should include the root_path
schema = sub_client.get("/openapi.json").json()
print("OpenAPI servers:", schema.get("servers", ["(not set — root_path only shown in live Swagger)"]))
print("Root path configured:", sub_path_app.root_path)

# Verify the X-Forwarded-* headers are forwarded correctly in tests
resp_with_headers = sub_client.get(
    "/health",
    headers={
        "X-Forwarded-For": "203.0.113.1",
        "X-Forwarded-Proto": "https",
    },
)
print("\nHealth (via proxy headers):", resp_with_headers.status_code, resp_with_headers.json())


**What just happened?**
- `root_path` tells FastAPI the prefix the reverse proxy is using — this makes the Swagger UI's "Try it out" work correctly when behind `/api/v1`
- Routes inside the app are still `/users`, `/health` — the proxy handles the prefix stripping
- **`X-Forwarded-*` headers** carry the real client IP and protocol through the proxy chain


---
## Challenge

Build a deployment-ready FastAPI app with the following:

1. A `Settings` class (pydantic-settings) with fields: `app_name`, `version`, `database_url`, `secret_key`, `debug`
2. Inject settings via `Depends(get_settings)` into at least two routes
3. A `/health` endpoint that returns `{"status": "ok", "version": "...", "debug": false}` — **503** if `debug=True` in production (pretend debug-mode is never ready)
4. Write 3 TestClient assertions:
   - `/health` returns 200 with `status == "ok"` when debug is False
   - Override settings to `debug=True` and verify `/health` returns 503
   - `/info` returns the `app_name` and `version`


In [ ]:
# Challenge: Build a deployment-ready FastAPI app
# Your solution here

# Scaffold:
from functools import lru_cache
from fastapi import FastAPI, Depends, Response, status
from fastapi.testclient import TestClient
from pydantic_settings import BaseSettings


class ChallengeSettings(BaseSettings):
    # TODO: add app_name, version, database_url, secret_key, debug fields
    pass


@lru_cache
def get_challenge_settings() -> ChallengeSettings:
    return ChallengeSettings()


challenge_app = FastAPI()

# TODO: add /health and /info routes
# TODO: write 3 TestClient assertions


---
## Day 13 key concepts recap

| Concept | What to remember |
|---|---|
| **Uvicorn** | ASGI server — run with `--host 0.0.0.0 --port 8000`; single worker in containers |
| **Gunicorn + UvicornWorker** | For VM deployments — `workers = 2 × cores + 1` |
| **pydantic-settings** | `BaseSettings` reads env vars + `.env` file automatically; validates types |
| **`@lru_cache` + `Depends`** | Cache settings singleton; inject via FastAPI dependency system |
| **`/health` endpoint** | Returns `200` when ready, `503` when not; checked by load balancers |
| **Dockerfile layers** | `COPY requirements.txt` → `pip install` → `COPY app/` — cheapest rebuild on code change |
| **Non-root user** | Security requirement — `adduser` + `USER appuser` in Dockerfile |
| **Exec form CMD** | `["uvicorn", ...]` not `"uvicorn ..."` — signals reach the process directly |
| **Reverse proxy** | Nginx / Traefik / ALB handles TLS; FastAPI handles `root_path` for sub-path deployments |

> **Tip:** For production: Gunicorn with UvicornWorker = workers = 2 × CPU cores + 1. For containers with autoscaling, run single-worker uvicorn.

---
## What's next
**Day 14** → Capstone: build a complete Bookmarks REST API with JWT auth, SQLAlchemy, full CRUD, tag filtering, and a pytest suite.

Mark Day 13 complete in your [tracker](../index.html).
